# Day 0: 分子生物学の最小基礎

## この章で解く現実の課題

炎症刺激を受けた細胞で `IL6` が増えた、と言うとき、細胞内では何が増えたと考えているのかを説明します。

データサイエンス的には、RNA-seqは「遺伝子 x サンプル」の表です。しかし、生物学的には、その表の値はDNA配列そのものではありません。細胞内でDNAからRNAへ読み出された分子の量を、readとして観測し、遺伝子ごとに集計したものです。


## 最小限の生物学

- DNA: 細胞が持つ設計図のような情報。基本的には細胞ごとに大きく変わらない。
- gene: DNA上の意味のある領域。タンパク質や機能性RNAの情報を持つ。
- transcription: DNAの情報をRNAとして写し取ること。
- RNA: いま細胞がどの遺伝子を使っているかを反映する分子。
- gene expression: ある遺伝子がRNAとしてどれくらい読み出されているか。

RNA-seqで見る「発現が高い」は、通常「その遺伝子由来のRNA断片が多く観測された」という意味です。DNAが増えた、遺伝子が増殖した、という意味ではありません。


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 生物学の言葉を、解析で見る表の列に対応づける。
# DNA配列そのものではなく、RNAとして読み出された量を扱う点が重要。
concept_map = pd.DataFrame({
    "biology_word": ["DNA", "gene", "RNA", "gene expression", "RNA-seq count"],
    "data_view": [
        "参照情報として使う設計図",
        "count matrix の行",
        "シーケンスされる分子の由来",
        "サンプル間で比較したい量",
        "遺伝子由来readを数えた観測値",
    ],
})
concept_map


## 具体例: IL6が増えた、を表で見る

`IL6` は炎症応答に関わる代表的な遺伝子です。以下では、対照群と刺激群で、いくつかの遺伝子由来read数がどう変わるかを小さな表で見ます。

ここで重要なのは、`IL6` の数値が増えるとは「IL6遺伝子から転写されたRNA由来のreadが多く観測された」という意味だということです。


In [ ]:
counts = pd.DataFrame({
    "gene": ["IL6", "TNF", "NFKBIA", "ACTB", "GAPDH"],
    "control_rep1": [8, 12, 30, 950, 870],
    "control_rep2": [10, 14, 28, 980, 910],
    "treated_rep1": [240, 180, 160, 1000, 890],
    "treated_rep2": [260, 210, 150, 970, 930],
}).set_index("gene")

# control平均とtreated平均を作る。
# ここでは生countを使って直感をつかむ。厳密な比較は次章以降で正規化してから行う。
summary = pd.DataFrame({
    "control_mean_count": counts[["control_rep1", "control_rep2"]].mean(axis=1),
    "treated_mean_count": counts[["treated_rep1", "treated_rep2"]].mean(axis=1),
})
summary["rough_fold_change"] = summary["treated_mean_count"] / summary["control_mean_count"]
summary.sort_values("rough_fold_change", ascending=False)


In [ ]:
ax = summary[["control_mean_count", "treated_mean_count"]].plot(kind="bar", figsize=(8, 4))
ax.set_ylabel("mean RNA-seq count")
ax.set_title("Inflammatory genes increase after stimulation")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 読み取り

`IL6`, `TNF`, `NFKBIA` は刺激後に大きく増えています。一方、`ACTB` や `GAPDH` はあまり変わっていません。後者はhousekeeping geneとして扱われることが多く、細胞の基本機能に関わるため条件間で大きく変わりにくい例として使えます。

ただし、ここではまだ正規化していません。RNA-seqでは、サンプルごとに読まれた総read数が違うため、生countだけで結論を出すのは危険です。Day 1とDay 2で、count matrixと正規化を扱います。


## エラーが出た場合

- `ModuleNotFoundError: No module named 'pandas'`: Colabなら通常入っています。ローカル実行の場合は環境が違う可能性があります。
- グラフが表示されない: セルを上から順番に実行してください。`summary` が未定義だとplotできません。

## この章のゴール

RNA-seqの発現量は、DNAの量ではなくRNAとして読み出された情報の観測値だと説明できれば合格です。
